# Lab 9: Sequence-to-Sequence Models in Python

## Step 1: Import libraries

In [ ]:
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## Step 2: Create the dataset

In [ ]:
formats = ['%d %B %Y', '%d %b %Y', '%B %d, %Y', '%d/%m/%Y', '%d.%m.%y', '%A, %d %B %Y', '%b %d %Y', '%d-%b-%y']

def random_date():
    d = datetime.date(1990, 1, 1) + datetime.timedelta(days=random.randint(0, 14600))
    return d.strftime(random.choice(formats)).lower(), d.isoformat()

data = [random_date() for _ in range(20000)]
df = pd.DataFrame(data, columns=['invoice_date', 'standard_date'])
print(df.head(8))
print(df['invoice_date'].str.len().describe())

## Step 3: Prepare the inputs

In [ ]:
in_chars = sorted(set(''.join(df['invoice_date'])))
out_chars = sorted(set(''.join(df['standard_date'])))
in_index = {c: i + 1 for i, c in enumerate(in_chars)}
out_index = {c: i + 1 for i, c in enumerate(out_chars)}
inv_out = {i: c for c, i in out_index.items()}
start_id = len(out_chars) + 1
in_vocab = len(in_chars) + 1
out_vocab = len(out_chars) + 2
print('Input vocabulary :', in_vocab)
print('Output vocabulary:', out_vocab)

max_in = 30
X = pad_sequences([[in_index[c] for c in s] for s in df['invoice_date']], maxlen=max_in, padding='post')
y = np.array([[out_index[c] for c in s] for s in df['standard_date']])
y_in = np.concatenate([np.full((len(y), 1), start_id), y[:, :-1]], axis=1)

X_train, X_test = X[:16000], X[16000:]
y_train, y_test = y[:16000], y[16000:]
y_in_train, y_in_test = y_in[:16000], y_in[16000:]
print(X_train.shape, y_in_train.shape, y_train.shape)
print(y[0])
print(y_in[0])

## Step 4: Build the encoder and decoder

In [ ]:
def build_seq2seq(units):
    enc_input = Input(shape=(max_in,), name='encoder_input')
    enc_emb = Embedding(in_vocab, 32, mask_zero=True)(enc_input)
    _, state_h, state_c = LSTM(units, return_state=True, name='encoder_lstm')(enc_emb)
    dec_input = Input(shape=(None,), name='decoder_input')
    dec_emb = Embedding(out_vocab, 32)(dec_input)
    dec_seq, _, _ = LSTM(units, return_sequences=True, return_state=True, name='decoder_lstm')(dec_emb, initial_state=[state_h, state_c])
    output = Dense(out_vocab, activation='softmax', name='output')(dec_seq)
    model = Model([enc_input, dec_input], output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

seq2seq = build_seq2seq(64)
seq2seq.summary()

## Step 5: Train the model

In [ ]:
history = seq2seq.fit(
    [X_train, y_in_train], y_train,
    epochs=15, batch_size=128,
    validation_split=0.1, verbose=2
)

plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Step 6: Generate and evaluate the output

In [ ]:
def translate(model, X):
    dec = np.full((len(X), 1), start_id)
    for _ in range(10):
        probs = model([X, dec], training=False).numpy()
        next_id = probs[:, -1, :].argmax(axis=-1)
        dec = np.concatenate([dec, next_id[:, None]], axis=1)
    return dec[:, 1:]

def to_text(ids):
    return ''.join(inv_out.get(i, '?') for i in ids)

pred = translate(seq2seq, X_test)
print('Exact-match accuracy:', round(np.mean(np.all(pred == y_test, axis=1)), 4))
print('Character accuracy  :', round(np.mean(pred == y_test), 4))

test_df = df.iloc[16000:].copy()
test_df['predicted'] = [to_text(p) for p in pred]
test_df['correct'] = test_df['predicted'] == test_df['standard_date']
print(test_df.sample(8, random_state=1))
print(test_df[~test_df['correct']].head(10))

## Step 7: See the bottleneck

In [ ]:
print('Context vector shape:', seq2seq.get_layer('encoder_lstm').output[1].shape)

small = build_seq2seq(16)
small.fit([X_train, y_in_train], y_train, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
small_pred = translate(small, X_test)
print('Exact-match accuracy with 16 units:', round(np.mean(np.all(small_pred == y_test, axis=1)), 4))

## Lab 9 Task 1: Own invoice dates

In [ ]:
my_dates = ['15 august 2021', 'mar 03 2015', '25/12/1999', '2018/09/07', '7th sep 2018']
my_true = ['2021-08-15', '2015-03-03', '1999-12-25', '2018-09-07', '2018-09-07']
my_X = pad_sequences([[in_index.get(c, 0) for c in s] for s in my_dates], maxlen=max_in, padding='post')
my_pred = [to_text(p) for p in translate(seq2seq, my_X)]
task1_df = pd.DataFrame({'Invoice date': my_dates, 'Expected': my_true, 'Predicted': my_pred})
task1_df['Correct'] = task1_df['Expected'] == task1_df['Predicted']
task1_df['Format seen in training'] = [True, True, True, False, False]
print(task1_df)

The model learned only the 8 formats in the training data. In '2018/09/07' the year comes first and is separated by '/', which in training always meant day/month/year, so the model tends to read the first digits as the day. In '7th sep 2018' the day has one digit and is followed by 'th', which never appeared in training. The model has no rule for these positions, so it often places digits in the wrong part of the output. A seq2seq model only generalises to patterns similar to its training data.

## Lab 9 Task 2: Effect of the number of units

In [ ]:
unit_results = []
for u in [8, 16, 32, 64, 128]:
    m = build_seq2seq(u)
    m.fit([X_train, y_in_train], y_train, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
    p = translate(m, X_test)
    unit_results.append({
        'Units': u,
        'Context vector size': 2 * u,
        'Parameters': m.count_params(),
        'Exact-match accuracy': round(float(np.mean(np.all(p == y_test, axis=1))), 4),
        'Character accuracy': round(float(np.mean(p == y_test)), 4)
    })
units_df = pd.DataFrame(unit_results)
print(units_df)

plt.figure(figsize=(8, 4))
plt.plot(units_df['Units'], units_df['Exact-match accuracy'], marker='o')
plt.xscale('log', base=2)
plt.xticks(units_df['Units'], units_df['Units'])
plt.title('Exact-match Accuracy vs Number of Units')
plt.xlabel('Units')
plt.ylabel('Exact-match accuracy')
plt.grid(True)
plt.show()

The encoder passes the whole input to the decoder only through state_h and state_c, so the context vector has 2 x units numbers. With 8 or 16 units this vector is too small to store the day, month and year together with their positions, and exact-match accuracy is low. As units increase the context vector can hold more information, and accuracy rises sharply up to 32 to 64 units and then levels off, while the parameter count keeps growing roughly with the square of the units. Beyond that point extra capacity gives little gain, which shows that the size of the fixed context vector is the main limit of the plain seq2seq model.

## Lab 9 Task 3: Accuracy by input length and format

In [ ]:
def find_format(s):
    for f in formats:
        try:
            datetime.datetime.strptime(s, f)
            return f
        except ValueError:
            pass
    return 'unknown'

def error_type(row):
    p, t = row['predicted'], row['standard_date']
    if p == t:
        return 'correct'
    if p[:4] == t[:4] and p[5:7] == t[8:10] and p[8:10] == t[5:7]:
        return 'day-month swap'
    parts = [name for name, a, b in [('year', p[:4], t[:4]), ('month', p[5:7], t[5:7]), ('day', p[8:10], t[8:10])] if a != b]
    return ' + '.join(parts) + ' wrong' if parts else 'separator wrong'

test_df['length'] = test_df['invoice_date'].str.len()
test_df['length_group'] = pd.cut(test_df['length'], bins=[0, 9, 12, 16, 20, 30], labels=['8-9', '10-12', '13-16', '17-20', '21-28'])
test_df['format'] = test_df['invoice_date'].apply(find_format)
test_df['error_type'] = test_df.apply(error_type, axis=1)

by_length = test_df.groupby('length_group', observed=True)['correct'].agg(['count', 'mean']).rename(columns={'count': 'Samples', 'mean': 'Exact-match accuracy'})
print(by_length.round(4))

by_format = test_df.groupby('format')['correct'].agg(['count', 'mean']).rename(columns={'count': 'Samples', 'mean': 'Exact-match accuracy'}).sort_values('Exact-match accuracy')
print(by_format.round(4))

errors = test_df[~test_df['correct']]
print(errors['error_type'].value_counts())
print(errors.groupby('format')['error_type'].agg(lambda s: s.value_counts().index[0]))

The numeric formats ('%d/%m/%Y' and '%d.%m.%y') usually cause the most errors, because the day and the month are both two digits and only their position separates them. The most common error type is a wrong day or a day-month swap, where the model knows which digits appear but puts them in the wrong place. Formats with a month name are easier, because the month is identified by letters that cannot be confused with the day. Long inputs with a weekday name also lose some accuracy, because more characters must pass through the same fixed-size context vector.

## Lab 9 Task 4: Business estimate

In [ ]:
best_acc = max(units_df['Exact-match accuracy'].max(), float(np.mean(np.all(pred == y_test, axis=1))))
monthly_invoices = 10000
wrong_per_month = round(monthly_invoices * (1 - best_acc))
print('Best exact-match accuracy :', round(best_acc, 4))
print('Expected wrong dates/month:', wrong_per_month, 'out of', monthly_invoices)

Even with the best model, a few hundred invoice dates per month would be wrong, and a wrong date can cause late payments, wrong due dates and errors in period-end reporting. Full automation is therefore not safe. The recommended approach is to use the model with human review: dates that fail a validity check, are numeric with a day value of 12 or less (where day and month could be swapped), or come in a new supplier format should go to an accounts-payable clerk. This removes most of the manual work while keeping control over the risky cases.

## Lab Assignment 3: Amounts in words to digits

### Task 1: Task description and example pairs

In [ ]:
random.seed(3)
np.random.seed(3)
tf.random.set_seed(3)

ones = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
        'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 'sixteen', 'seventeen', 'eighteen', 'nineteen']
tens = ['', '', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety']

def below_thousand(n, use_and):
    parts = []
    if n >= 100:
        parts += [ones[n // 100], 'hundred']
        n %= 100
        if n and use_and:
            parts.append('and')
    if n >= 20:
        parts.append(tens[n // 10])
        n %= 10
        if n:
            parts.append(ones[n])
    elif n > 0:
        parts.append(ones[n])
    return parts

def number_to_words(n):
    use_and = random.random() < 0.3
    parts = []
    if n >= 1000:
        parts += below_thousand(n // 1000, use_and) + ['thousand']
        n %= 1000
    if n:
        parts += below_thousand(n, use_and)
    return ' '.join(parts)

numbers = random.sample(range(1, 10000), 3000) + random.sample(range(10000, 1000000), 9000)
random.shuffle(numbers)
amount_df = pd.DataFrame({'amount_words': [number_to_words(n) for n in numbers], 'amount_digits': [str(n) for n in numbers]})
print(amount_df.head(10).to_string())
print(len(amount_df), 'pairs')
print(amount_df['amount_words'].str.len().describe())

Cheques, invoices and contracts often write amounts in words ("two thousand five hundred"), and finance teams must convert them into digits before they can be checked against the amount written in figures or entered into the accounting system. A rule-based converter must handle every word for units, teens, tens, hundreds and thousands, optional words such as "and", different spellings and spacing, and the place value that each word implies. Every new variation needs new rules and new exceptions, so the code becomes long and fragile. A seq2seq model learns these place-value rules directly from examples and can be retrained when new patterns appear.

### Task 2: Encoder input, decoder input and decoder target

In [ ]:
a_out_len = 7
a_in_chars = sorted(set(''.join(amount_df['amount_words'])))
a_out_chars = sorted(set('0123456789$'))
a_in_index = {c: i + 1 for i, c in enumerate(a_in_chars)}
a_out_index = {c: i + 1 for i, c in enumerate(a_out_chars)}
a_inv_out = {i: c for c, i in a_out_index.items()}
a_start = len(a_out_chars) + 1
a_in_vocab = len(a_in_chars) + 1
a_out_vocab = len(a_out_chars) + 2
a_max_in = int(amount_df['amount_words'].str.len().max()) + 2

a_X = pad_sequences([[a_in_index[c] for c in s] for s in amount_df['amount_words']], maxlen=a_max_in, padding='post')
a_targets = [s + '$' * (a_out_len - len(s)) for s in amount_df['amount_digits']]
a_y = np.array([[a_out_index[c] for c in s] for s in a_targets])
a_y_in = np.concatenate([np.full((len(a_y), 1), a_start), a_y[:, :-1]], axis=1)

a_X_train, a_X_test = a_X[:10000], a_X[10000:]
a_y_train, a_y_test = a_y[:10000], a_y[10000:]
a_y_in_train, a_y_in_test = a_y_in[:10000], a_y_in[10000:]

print('Input vocabulary :', a_in_vocab)
print('Output vocabulary:', a_out_vocab)
print('Encoder input  :', a_X_train.shape)
print('Decoder input  :', a_y_in_train.shape)
print('Decoder target :', a_y_train.shape)
print(amount_df['amount_digits'].iloc[0], a_targets[0])
print(a_y[0])
print(a_y_in[0])

### Task 3: Build and train the encoder-decoder LSTM

In [ ]:
def build_amount_model(units):
    enc_input = Input(shape=(a_max_in,), name='encoder_input')
    enc_emb = Embedding(a_in_vocab, 32, mask_zero=True, name='encoder_embedding')(enc_input)
    _, state_h, state_c = LSTM(units, return_state=True, name='encoder_lstm')(enc_emb)
    dec_input = Input(shape=(None,), name='decoder_input')
    dec_emb = Embedding(a_out_vocab, 32, name='decoder_embedding')(dec_input)
    dec_seq, _, _ = LSTM(units, return_sequences=True, return_state=True, name='decoder_lstm')(dec_emb, initial_state=[state_h, state_c])
    output = Dense(a_out_vocab, activation='softmax', name='output')(dec_seq)
    model = Model([enc_input, dec_input], output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def translate_amount(model, X):
    dec = np.full((len(X), 1), a_start)
    for _ in range(a_out_len):
        probs = model([X, dec], training=False).numpy()
        dec = np.concatenate([dec, probs[:, -1, :].argmax(axis=-1)[:, None]], axis=1)
    return dec[:, 1:]

def amount_text(ids):
    return ''.join(a_inv_out.get(i, '?') for i in ids).replace('$', '')

amount_model = build_amount_model(128)
amount_model.summary()
amount_history = amount_model.fit([a_X_train, a_y_in_train], a_y_train, epochs=30, batch_size=128, validation_split=0.1, verbose=2)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(amount_history.history['loss'], label='Training loss')
plt.plot(amount_history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(amount_history.history['accuracy'], label='Training accuracy')
plt.plot(amount_history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

- **encoder_input**: receives the amount in words as a padded sequence of character ids.
- **encoder_embedding**: turns every character id into a 32-number vector; mask_zero=True makes the LSTM skip the padding.
- **encoder_lstm**: reads the characters one by one and keeps only its final hidden state and cell state, which together form the context vector that summarises the whole amount.
- **decoder_input**: receives the start token followed by the correct previous digits (teacher forcing).
- **decoder_embedding**: turns every output id into a 32-number vector.
- **decoder_lstm**: starts from the context vector and produces one hidden state for each of the 7 output positions.
- **output (Dense + softmax)**: gives a probability for each of the 13 output symbols (digits, the end/padding symbol '$', padding id and start token) at every position.

### Task 4: Exact-match and character accuracy

In [ ]:
a_pred = translate_amount(amount_model, a_X_test)
print('Exact-match accuracy:', round(float(np.mean(np.all(a_pred == a_y_test, axis=1))), 4))
print('Character accuracy  :', round(float(np.mean(a_pred == a_y_test)), 4))

amount_test = amount_df.iloc[10000:].copy()
amount_test['predicted'] = [amount_text(p) for p in a_pred]
amount_test['correct'] = amount_test['predicted'] == amount_test['amount_digits']
print(amount_test.sample(10, random_state=1).to_string())
print(amount_test[~amount_test['correct']].head(10).to_string())

### Task 5: Size of the context vector

In [ ]:
amount_units = []
for u in [32, 64, 128, 256]:
    if u == 128:
        p = a_pred
        params = amount_model.count_params()
    else:
        m = build_amount_model(u)
        m.fit([a_X_train, a_y_in_train], a_y_train, epochs=30, batch_size=128, validation_split=0.1, verbose=0)
        p = translate_amount(m, a_X_test)
        params = m.count_params()
    amount_units.append({
        'Units': u,
        'Context vector size': 2 * u,
        'Parameters': params,
        'Exact-match accuracy': round(float(np.mean(np.all(p == a_y_test, axis=1))), 4),
        'Character accuracy': round(float(np.mean(p == a_y_test)), 4)
    })
amount_units_df = pd.DataFrame(amount_units)
print(amount_units_df)

plt.figure(figsize=(8, 4))
plt.plot(amount_units_df['Units'], amount_units_df['Exact-match accuracy'], marker='o', label='Exact-match')
plt.plot(amount_units_df['Units'], amount_units_df['Character accuracy'], marker='s', label='Character')
plt.xscale('log', base=2)
plt.xticks(amount_units_df['Units'], amount_units_df['Units'])
plt.xlabel('Units')
plt.ylabel('Accuracy')
plt.title('Amount Conversion: Effect of Context Vector Size')
plt.legend()
plt.grid(True)
plt.show()

### Task 6: Conclusion

A finance department can use this model to read the amount in words on cheques, invoices, purchase orders and contracts and convert it into digits automatically. The converted value can then be compared with the amount written in figures, so that any mismatch is flagged before payment, which is a standard control against fraud and data-entry mistakes. The model removes repetitive manual typing and can be retrained when new wording appears, without rewriting rules.

The risk of a wrong output is high, because an amount with one wrong or missing digit can be off by a factor of ten or more, leading to overpayment, underpayment, failed reconciliations or misstated accounts. The experiments show that a larger context vector improves exact-match accuracy, but long amounts with many words remain harder because they must pass through one fixed vector. For this reason the model should not post payments by itself: every converted amount should be checked against the figure written in digits, and any mismatch or unusual amount should be sent to a person for approval. Used this way, the model saves time while the business keeps control over financial risk.

# Lab 10: Seq2Seq with Attention vs without Attention

## Step 1: Compute attention by hand

In [ ]:
import numpy as np

words = ['the', 'delivery', 'was', 'very', 'late']
encoder_states = np.array([
    [0.1, 0.0, 0.2],
    [0.9, 0.1, 0.3],
    [0.0, 0.1, 0.1],
    [0.2, 0.6, 0.1],
    [0.8, 0.9, 0.2]
])
decoder_state = np.array([1.0, 1.0, 0.0])

scores = encoder_states @ decoder_state
weights = np.exp(scores) / np.exp(scores).sum()
context = weights @ encoder_states
for w, s, a in zip(words, scores, weights):
    print(f'{w:10s} score={s:.2f} weight={a:.2f}')
print('Context vector:', context.round(3))

## Step 2: Import libraries and recreate the data

In [ ]:
import time
import random
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Attention, Concatenate
from tensorflow.keras.preprocessing.sequence import pad_sequences

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

formats = ['%d %B %Y', '%d %b %Y', '%B %d, %Y', '%d/%m/%Y', '%d.%m.%y', '%A, %d %B %Y', '%b %d %Y', '%d-%b-%y']

def random_date():
    d = datetime.date(1990, 1, 1) + datetime.timedelta(days=random.randint(0, 14600))
    return d.strftime(random.choice(formats)).lower(), d.isoformat()

df = pd.DataFrame([random_date() for _ in range(20000)], columns=['invoice_date', 'standard_date'])

in_chars = sorted(set(''.join(df['invoice_date'])))
out_chars = sorted(set(''.join(df['standard_date'])))
in_index = {c: i + 1 for i, c in enumerate(in_chars)}
out_index = {c: i + 1 for i, c in enumerate(out_chars)}
inv_out = {i: c for c, i in out_index.items()}
start_id = len(out_chars) + 1
in_vocab = len(in_chars) + 1
out_vocab = len(out_chars) + 2

max_in = 30
X = pad_sequences([[in_index[c] for c in s] for s in df['invoice_date']], maxlen=max_in, padding='post')
y = np.array([[out_index[c] for c in s] for s in df['standard_date']])
y_in = np.concatenate([np.full((len(y), 1), start_id), y[:, :-1]], axis=1)

X_train, X_test = X[:16000], X[16000:]
y_train, y_test = y[:16000], y[16000:]
y_in_train, y_in_test = y_in[:16000], y_in[16000:]

def translate(model, X):
    dec = np.full((len(X), 1), start_id)
    for _ in range(10):
        probs = model([X, dec], training=False).numpy()
        next_id = probs[:, -1, :].argmax(axis=-1)
        dec = np.concatenate([dec, next_id[:, None]], axis=1)
    return dec[:, 1:]

def to_text(ids):
    return ''.join(inv_out.get(i, '?') for i in ids)

## Step 3: Define one function for both models

In [ ]:
units = 32

def build_model(use_attention):
    enc_input = Input(shape=(max_in,), name='encoder_input')
    enc_emb = Embedding(in_vocab, 32, mask_zero=True)(enc_input)
    enc_seq, state_h, state_c = LSTM(units, return_sequences=True, return_state=True, name='encoder_lstm')(enc_emb)
    dec_input = Input(shape=(None,), name='decoder_input')
    dec_emb = Embedding(out_vocab, 32)(dec_input)
    dec_seq, _, _ = LSTM(units, return_sequences=True, return_state=True, name='decoder_lstm')(dec_emb, initial_state=[state_h, state_c])
    score_model = None
    if use_attention:
        context, scores = Attention(name='attention')([dec_seq, enc_seq], return_attention_scores=True)
        dec_seq = Concatenate(name='concat')([dec_seq, context])
        score_model = Model([enc_input, dec_input], scores)
    output = Dense(out_vocab, activation='softmax', name='output')(dec_seq)
    model = Model([enc_input, dec_input], output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, score_model

base_model, _ = build_model(use_attention=False)
attn_model, score_model = build_model(use_attention=True)
base_model.summary()
attn_model.summary()

## Step 4: Train both models

In [ ]:
def train(model):
    start = time.time()
    history = model.fit(
        [X_train, y_in_train], y_train,
        epochs=20, batch_size=128,
        validation_split=0.1, verbose=2
    )
    return history, time.time() - start

base_history, base_time = train(base_model)
attn_history, attn_time = train(attn_model)

## Step 5: Compare the results

In [ ]:
base_pred = translate(base_model, X_test)
attn_pred = translate(attn_model, X_test)

results = pd.DataFrame({
    'Model': ['Seq2Seq', 'Seq2Seq + Attention'],
    'Parameters': [base_model.count_params(), attn_model.count_params()],
    'Exact-match accuracy': [np.mean(np.all(base_pred == y_test, axis=1)).round(4), np.mean(np.all(attn_pred == y_test, axis=1)).round(4)],
    'Character accuracy': [np.mean(base_pred == y_test).round(4), np.mean(attn_pred == y_test).round(4)],
    'Training time (s)': [round(base_time, 1), round(attn_time, 1)]
})
print(results)

plt.figure(figsize=(10, 4))
plt.plot(base_history.history['val_accuracy'], marker='o', label='Seq2Seq')
plt.plot(attn_history.history['val_accuracy'], marker='o', label='Seq2Seq + Attention')
plt.title('Validation Accuracy: With and Without Attention')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Step 6: See where the model looks

In [ ]:
def show_attention(i):
    text = df['invoice_date'].iloc[16000 + i]
    out_ids = translate(attn_model, X_test[i:i + 1])[0]
    dec = np.concatenate([[start_id], out_ids[:-1]])[None, :]
    weights = score_model([X_test[i:i + 1], dec], training=False).numpy()[0][:, :len(text)]
    plt.figure(figsize=(8, 5))
    plt.imshow(weights, cmap='Blues')
    plt.xticks(range(len(text)), list(text))
    plt.yticks(range(10), list(to_text(out_ids)))
    plt.xlabel('Input: invoice date')
    plt.ylabel('Output: standard date')
    plt.colorbar()
    plt.show()

show_attention(0)
show_attention(5)

## Step 7: Test on your own invoice dates

In [ ]:
def standardise(texts, model):
    ids = pad_sequences([[in_index.get(c, 0) for c in t.lower()] for t in texts], maxlen=max_in, padding='post')
    return [to_text(p) for p in translate(model, ids)]

samples = ['3 march 2021', 'dec 25 2019', '01/07/2024', 'friday, 14 june 2002']
print(pd.DataFrame({
    'Input': samples,
    'Seq2Seq': standardise(samples, base_model),
    'Seq2Seq + Attention': standardise(samples, attn_model)
}))

## Lab 10 Task 1: Attention heatmaps for three formats

In [ ]:
from tensorflow.keras.layers import AdditiveAttention

def find_format(s):
    for f in formats:
        try:
            datetime.datetime.strptime(s, f)
            return f
        except ValueError:
            pass
    return 'unknown'

def translate_gen(model, X, start, steps):
    dec = np.full((len(X), 1), start)
    for _ in range(steps):
        probs = model([X, dec], training=False).numpy()
        dec = np.concatenate([dec, probs[:, -1, :].argmax(axis=-1)[:, None]], axis=1)
    return dec[:, 1:]

def plot_attention(model, smodel, x_row, text, start, steps, idx_to_char, title=''):
    out_ids = translate_gen(model, x_row[None, :], start, steps)[0]
    dec = np.concatenate([[start], out_ids[:-1]])[None, :]
    w = smodel([x_row[None, :], dec], training=False).numpy()[0][:, :len(text)]
    out_text = ''.join(idx_to_char.get(i, '?') for i in out_ids)
    plt.figure(figsize=(max(8, len(text) * 0.32), 5))
    plt.imshow(w, cmap='Blues', aspect='auto')
    plt.xticks(range(len(text)), list(text))
    plt.yticks(range(steps), list(out_text))
    plt.title(title)
    plt.xlabel('Input')
    plt.ylabel('Output')
    plt.colorbar()
    plt.show()

lab10_test = df.iloc[16000:].copy().reset_index(drop=True)
lab10_test['format'] = lab10_test['invoice_date'].apply(find_format)

for f in ['%d/%m/%Y', '%B %d, %Y', '%A, %d %B %Y']:
    i = int(np.where(lab10_test['format'].values == f)[0][0])
    plot_attention(attn_model, score_model, X_test[i], lab10_test['invoice_date'].iloc[i], start_id, 10, inv_out,
                   title=f"{lab10_test['invoice_date'].iloc[i]}  ->  {lab10_test['standard_date'].iloc[i]}")

wk_idx = np.where(lab10_test['format'].values == '%A, %d %B %Y')[0][:300]
wk_w = score_model([X_test[wk_idx], y_in_test[wk_idx]], training=False).numpy()
wk_mass = [wk_w[k][:, :lab10_test['invoice_date'].iloc[j].index(',')].sum(axis=1).mean() for k, j in enumerate(wk_idx)]
print('Average share of attention on the weekday name:', round(float(np.mean(wk_mass)), 3))

- **dd/mm/yyyy (e.g. 26/12/2000)**: when writing the year the model looks at the last four digits, when writing the month it looks at the two digits after the first '/', and when writing the day it looks at the first two digits.
- **Month name first (e.g. april 06, 1992)**: the year comes from the four digits after the comma, the month from the letters of the month name (mainly its first letters), and the day from the two digits before the comma.
- **Weekday format (e.g. friday, 14 june 2002)**: the weights fall on the day digits, the month word and the year digits. The weekday name before the comma receives almost no weight (see the printed share above), so the model has learned to ignore it, because the weekday adds no information that is not already in the date.

## Lab 10 Task 2: Additive (Bahdanau) attention

In [ ]:
def build_any(kind, n_units, max_len, in_v, out_v):
    enc_input = Input(shape=(max_len,), name='encoder_input')
    enc_emb = Embedding(in_v, 32, mask_zero=True)(enc_input)
    enc_seq, state_h, state_c = LSTM(n_units, return_sequences=True, return_state=True, name='encoder_lstm')(enc_emb)
    dec_input = Input(shape=(None,), name='decoder_input')
    dec_emb = Embedding(out_v, 32)(dec_input)
    dec_seq, _, _ = LSTM(n_units, return_sequences=True, return_state=True, name='decoder_lstm')(dec_emb, initial_state=[state_h, state_c])
    score_model = None
    if kind != 'none':
        layer = Attention(name='attention') if kind == 'dot' else AdditiveAttention(name='attention')
        context, scores = layer([dec_seq, enc_seq], return_attention_scores=True)
        dec_seq = Concatenate(name='concat')([dec_seq, context])
        score_model = Model([enc_input, dec_input], scores)
    output = Dense(out_v, activation='softmax', name='output')(dec_seq)
    model = Model([enc_input, dec_input], output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model, score_model

add_model, add_score = build_any('additive', units, max_in, in_vocab, out_vocab)
add_model.summary()
add_history, add_time = train(add_model)
add_pred = translate(add_model, X_test)

results.loc[len(results)] = ['Seq2Seq + Additive Attention', add_model.count_params(),
                             np.mean(np.all(add_pred == y_test, axis=1)).round(4),
                             np.mean(add_pred == y_test).round(4), round(add_time, 1)]

def sharpness(smodel, n=500):
    w = smodel([X_test[:n], y_in_test[:n]], training=False).numpy()
    return round(float(w.max(axis=-1).mean()), 3)

results['Mean max attention weight'] = [np.nan, sharpness(score_model), sharpness(add_score)]
print(results)

for i in [0, 5]:
    plot_attention(attn_model, score_model, X_test[i], lab10_test['invoice_date'].iloc[i], start_id, 10, inv_out, title='Dot-product attention')
    plot_attention(add_model, add_score, X_test[i], lab10_test['invoice_date'].iloc[i], start_id, 10, inv_out, title='Additive attention')

Additive attention scores each pair of decoder and encoder states with a small learned network, so it adds a few parameters, while dot-product attention has none. Both reach a similar, much higher exact-match accuracy than the model without attention. The mean max attention weight measures how sharp the heatmaps are: a value near 1 means the model puts almost all of its attention on one input character. The model with the higher value gives cleaner, more focused heatmaps, which are easier to explain; the other spreads its weight over neighbouring characters but still focuses on the right segment of the date.

## Lab 10 Task 3: 16, 32 and 64 units

In [ ]:
size_rows = []
for u in [16, 32, 64]:
    row = {'Units': u}
    for kind, label in [('none', 'Seq2Seq'), ('dot', 'Seq2Seq + Attention')]:
        if u == 32:
            p = base_pred if kind == 'none' else attn_pred
        else:
            m, _ = build_any(kind, u, max_in, in_vocab, out_vocab)
            m.fit([X_train, y_in_train], y_train, epochs=20, batch_size=128, validation_split=0.1, verbose=0)
            p = translate(m, X_test)
        row[label] = round(float(np.mean(np.all(p == y_test, axis=1))), 4)
    size_rows.append(row)
size_df = pd.DataFrame(size_rows)
size_df['Gap'] = (size_df['Seq2Seq + Attention'] - size_df['Seq2Seq']).round(4)
print(size_df)

plt.figure(figsize=(8, 4))
plt.plot(size_df['Units'], size_df['Seq2Seq'], marker='o', label='Seq2Seq')
plt.plot(size_df['Units'], size_df['Seq2Seq + Attention'], marker='o', label='Seq2Seq + Attention')
plt.xticks(size_df['Units'])
plt.title('Exact-match Accuracy vs Units')
plt.xlabel('Units')
plt.ylabel('Exact-match accuracy')
plt.legend()
plt.grid(True)
plt.show()

Without attention the decoder sees only the final encoder state, so everything about the input must fit into 2 x units numbers. With 16 units this context vector is far too small and accuracy drops sharply, and it only recovers when the vector becomes large (64 units). With attention the decoder can read all 30 encoder states again at every output step, so it does not depend on one fixed summary. Its accuracy stays high even at 16 units, and the gap between the two models is largest for the smallest model and narrows as the context vector grows. This shows that attention removes the context-vector bottleneck from Lab 9.

## Lab 10 Task 4: Recommendation

In [ ]:
monthly = 50000
rec = results[['Model', 'Exact-match accuracy', 'Training time (s)']].copy()
rec['Wrong dates per month'] = ((1 - rec['Exact-match accuracy'].astype(float)) * monthly).round().astype(int)
print(rec)

The recommended model is seq2seq with dot-product attention. At 50,000 invoices a month it produces far fewer wrong dates than the model without attention (see the table above), which directly reduces payment errors and manual correction work. It adds only about 416 parameters and a small increase in training time, which is negligible because the model is trained once and then used many times. Additive attention gives similar accuracy but needs extra parameters and is slower. Most importantly, the attention heatmaps show which characters were used for the day, month and year, so auditors can check why a particular date was produced. Outputs that fail a date validity check should still be sent for human review.

## Lab Assignment 4: Attention-based sequence model comparison study

### Task 1: Three models under the same settings (standard invoice dates)

In [ ]:
short_rows = []
for name, m, p, t in [('Seq2Seq', base_model, base_pred, base_time),
                      ('Seq2Seq + Dot Attention', attn_model, attn_pred, attn_time),
                      ('Seq2Seq + Additive Attention', add_model, add_pred, add_time)]:
    short_rows.append({
        'Input': 'Date only',
        'Model': name,
        'Parameters': m.count_params(),
        'Exact-match accuracy': round(float(np.mean(np.all(p == y_test, axis=1))), 4),
        'Character accuracy': round(float(np.mean(p == y_test)), 4),
        'Training time (s)': round(t, 1)
    })
print(pd.DataFrame(short_rows))

### Task 2: Longer inputs (supplier name and invoice number before the date)

In [ ]:
random.seed(7)
np.random.seed(7)
tf.random.set_seed(7)

suppliers = ['al-falah traders', 'karachi steel works', 'nishat textiles', 'zamzam foods', 'quetta marble co',
             'packages ltd', 'lucky cement', 'engro fertilizers', 'habib oil mills', 'sapphire fibres',
             'hashoo logistics', 'kohinoor mills']
templates = ['{s} inv-{n} {d}', '{s} | invoice #{n} | {d}', 'invoice {n} from {s} dated {d}', '{s}, bill no {n}, {d}']

long_df = pd.DataFrame({
    'invoice_text': [random.choice(templates).format(s=random.choice(suppliers), n=random.randint(1000, 99999), d=d) for d in df['invoice_date']],
    'standard_date': df['standard_date']
})
print(long_df.head(8).to_string())
print(long_df['invoice_text'].str.len().describe())

l_chars = sorted(set(''.join(long_df['invoice_text'])))
l_index = {c: i + 1 for i, c in enumerate(l_chars)}
l_vocab = len(l_chars) + 1
max_long = int(long_df['invoice_text'].str.len().max()) + 2

XL = pad_sequences([[l_index[c] for c in s] for s in long_df['invoice_text']], maxlen=max_long, padding='post')
XL_train, XL_test = XL[:16000], XL[16000:]
print('Input vocabulary:', l_vocab, ' max length:', max_long, ' shapes:', XL_train.shape, XL_test.shape)

def train_long(model):
    start = time.time()
    history = model.fit([XL_train, y_in_train], y_train, epochs=20, batch_size=128, validation_split=0.1, verbose=2)
    return history, time.time() - start

long_models = {}
for kind, name in [('none', 'Seq2Seq'), ('dot', 'Seq2Seq + Dot Attention'), ('additive', 'Seq2Seq + Additive Attention')]:
    m, sm = build_any(kind, units, max_long, l_vocab, out_vocab)
    h, t = train_long(m)
    p = translate_gen(m, XL_test, start_id, 10)
    long_models[name] = {'model': m, 'score_model': sm, 'history': h, 'time': t, 'pred': p}

plt.figure(figsize=(10, 4))
for name, r in long_models.items():
    plt.plot(r['history'].history['val_accuracy'], marker='o', label=name)
plt.title('Validation Accuracy on Long Inputs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### Task 3: Results table

In [ ]:
long_rows = []
for name, r in long_models.items():
    long_rows.append({
        'Input': 'Supplier + invoice no + date',
        'Model': name,
        'Parameters': r['model'].count_params(),
        'Exact-match accuracy': round(float(np.mean(np.all(r['pred'] == y_test, axis=1))), 4),
        'Character accuracy': round(float(np.mean(r['pred'] == y_test)), 4),
        'Training time (s)': round(r['time'], 1)
    })
study = pd.DataFrame(short_rows + long_rows)
print(study.to_string(index=False))

pivot = study.pivot(index='Model', columns='Input', values='Exact-match accuracy')
pivot.plot(kind='bar', figsize=(9, 4), rot=0)
plt.title('Exact-match Accuracy: Short vs Long Inputs')
plt.ylabel('Exact-match accuracy')
plt.ylim(0, 1.05)
plt.show()

### Task 4: Heatmaps for correct and wrong predictions of the best model

In [ ]:
best_name = max(['Seq2Seq + Dot Attention', 'Seq2Seq + Additive Attention'],
                key=lambda n: np.mean(np.all(long_models[n]['pred'] == y_test, axis=1)))
best = long_models[best_name]
print('Best model:', best_name)

long_test = long_df.iloc[16000:].copy().reset_index(drop=True)
long_test['predicted'] = [to_text(p) for p in best['pred']]
long_test['base_predicted'] = [to_text(p) for p in long_models['Seq2Seq']['pred']]
long_test['correct'] = long_test['predicted'] == long_test['standard_date']
long_test['base_correct'] = long_test['base_predicted'] == long_test['standard_date']

correct_idx = np.where(long_test['correct'])[0][:2]
wrong_idx = np.where(~long_test['correct'])[0][:2]
print(long_test.loc[wrong_idx, ['invoice_text', 'standard_date', 'predicted']].to_string())

for i in list(correct_idx) + list(wrong_idx):
    status = 'CORRECT' if long_test['correct'].iloc[i] else 'WRONG'
    plot_attention(best['model'], best['score_model'], XL_test[i], long_test['invoice_text'].iloc[i], start_id, 10, inv_out,
                   title=f"{status}: true {long_test['standard_date'].iloc[i]}, predicted {long_test['predicted'].iloc[i]}")

In the correct examples the attention falls only on the date at the end of the string; the supplier name and the invoice number receive almost no weight, so the model has learned to locate the date inside the longer text. In the wrong examples the attention for one or more output digits is spread out or lands on the wrong characters, for example on digits of the invoice number instead of the day or year, or it swaps the day and month positions in a numeric date. The heatmap shows exactly which output character went wrong and where the model looked, which is what an analyst needs to diagnose and fix the error.

### Task 5: Inputs that only the attention model got right

In [ ]:
only_attn = long_test[(~long_test['base_correct']) & (long_test['correct'])]
print('Long inputs wrong without attention but right with attention:', len(only_attn))
print(only_attn[['invoice_text', 'standard_date', 'base_predicted', 'predicted']].head(5).to_string())

short_test = df.iloc[16000:].copy().reset_index(drop=True)
short_test['base_predicted'] = [to_text(p) for p in base_pred]
short_test['attn_predicted'] = [to_text(p) for p in attn_pred]
only_attn_short = short_test[(short_test['base_predicted'] != short_test['standard_date']) & (short_test['attn_predicted'] == short_test['standard_date'])]
print('Short inputs wrong without attention but right with attention:', len(only_attn_short))
print(only_attn_short.head(5).to_string())

for i in only_attn.index[:2]:
    plot_attention(best['model'], best['score_model'], XL_test[i], long_test['invoice_text'].iloc[i], start_id, 10, inv_out,
                   title=f"Attention right ({long_test['predicted'].iloc[i]}), no attention wrong ({long_test['base_predicted'].iloc[i]})")

These inputs are usually the longest ones, with a long supplier name, a five-digit invoice number and a long date format, or numeric dates where the day and month are easy to confuse. Without attention the encoder must keep the date digits in its final state after reading 50 to 70 characters, and the invoice-number digits and the end of the string overwrite part of that information, so the model copies wrong digits or swaps the day and month. With attention the decoder can look directly at the exact characters of the day, month and year when it writes each of them, as the heatmaps show, so the length of the text in front of the date does not matter much.

### Task 6: Conclusion

The model to deploy is the seq2seq model with attention (the best of the two attention models in the table above, normally dot-product attention because it has no extra parameters and is faster). On the plain invoice dates all models do reasonably well, but attention already gives a higher exact-match accuracy with the same small size. When the date is hidden inside a longer string with a supplier name and an invoice number, the model without attention loses much more accuracy, because the whole string must pass through one fixed context vector of 64 numbers. The attention models keep a high accuracy because they rebuild the context at every output step from all encoder states and simply ignore the irrelevant parts of the text.

Attention also helps beyond accuracy. The heatmaps show which characters were used for each output digit, so wrong predictions can be diagnosed and auditors can see why the system produced a particular date. The cost is small: only a few hundred extra parameters and a modest increase in training time. Its limits are that it still fails on formats that never appeared in training and occasionally attends to the wrong digits, so outputs should be validated and uncertain cases reviewed by a person.

The same idea is the core of Transformer models such as BERT and GPT. A Transformer removes the LSTM completely and uses scaled dot-product attention between every pair of tokens (self-attention), with many attention heads working in parallel and stacked in layers. Each token builds its representation as a weighted average of all other tokens, exactly like the context vector in this lab. Because there is no step-by-step recurrence, Transformers can be trained in parallel on very large datasets, which is why they power modern translation systems, document-processing tools and chatbots.